In [ ]:
%run fabric-workspace-delete-util

In [ ]:
def create_new_workspace(display_name_in, capacity_id, description_in):
    ## example, using capacity: create_new_workspace("ABC", common_capacity, "ABC123")
    ## example, Pro workspace: create_new_workspace("DEF", None, "DEF456")

    # requires Fabric Admin rights to successfully call the workspace membership update components
    # encapsulated Semantic Labs function so that we can do the workspace provisioning and also auto-assign Fabric administrative group to the target workspace
    new_workspace_id = fabric.create_workspace(display_name_in, capacity_id, description_in)
    ## add the Fabric admins to the workspace

    '''
    ------------------- READ THIS COMMENT --------------------------
    
    Because there is currently no Service Principal support for the Admin APIS `labs.admin.delete_workspace()`, we are forced to do this component outside of SP contexts.
    Have raised a feature enhancement request on SLL github: https://github.com/microsoft/semantic-link-labs/issues/1104

    If the SP context for AddUser / DeleteUser (from Workspace) Admin APIs is added, we can then indent this entire function under 1x `with labs.service_principal_authentication(...)` block
    Will also be able to add SP intent to delete_workspace()

    -------------- THANK YOU FOR YOUR ATTENTION --------------------

    '''

    # adding conditional logic as not all necessary Admin APIs currently support Service Principal authentication - in future we can remove this check 
    if orchestration_sp_app_object_id == executing_user: 
        # Use SP context - requires Keyvault access to read secrets(PIM or via SP caller)
        with labs.service_principal_authentication(
            key_vault_uri = key_vault_uri,
            key_vault_tenant_id = 'tenant-fabric-sp',
            key_vault_client_id = 'fabric-admin-sp-app-id', 
            key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
            labs.admin.add_user_to_workspace(
                fabric_admins,
                'Admin',
                'Group',
                new_workspace_id
            )
            ## remove the notebook executing user from the workspace
            labs.admin.delete_user_from_workspace(executing_user, new_workspace_id)
            ## End Service Principal context
    else:
    # use User context (requires Fabric Admin)
        labs.admin.add_user_to_workspace(
                fabric_admins,
                'Admin',
                'Group',
                new_workspace_id
            )
        ## remove the notebook executing user from the workspace
        labs.admin.delete_user_from_workspace(executing_user, new_workspace_id)
        ## End user context    

    # define a dictionary of the key identifiers so they can be passed to related / upstream functions
    new_workspace_details = {
        'new_workspace_id': new_workspace_id,
        'new_workspace_name': display_name_in
    }
    # confirm successful execution
    print("Created new Workspace: " + display_name_in + " as " + new_workspace_id)
    return new_workspace_details

In [ ]:
# code snippet for testing
## Start test
try: 
    new_ws = create_new_workspace('testname', None, "test desc")
    delete_workspace_access(new_ws['new_workspace_id'])
except:
    create_workspace_error_dict = {
        "message": "Create workspace function has an issue! Check Fabric Admin authorisation"
    }
    mssparkutils.notebook.exit(create_workspace_error_dict)
## End test